In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

In [ ]:
import os
import json
from datetime import datetime
from bs4 import BeautifulSoup
import time

from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
import re
import torch

model = MBartForConditionalGeneration.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
tokenizer = MBart50TokenizerFast.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model.to(device)

tokenizer.src_lang = "ne_XX"
tokenizer.tgt_lang = "en_XX"

def remove_html(text):
        soup = BeautifulSoup(text, "html.parser")
        return soup.get_text(separator=' ', strip=True)




In [ ]:

with open("tmp.json", "r", encoding="utf-8") as f:
    data = json.load(f)
translated_list = []

for item in data:
    item["content"] = remove_html(item["content"])
    item["title"] = remove_html(item["title"])
    print(item["title"])

    text = item["title"] + " " +  item["content"]
    inputs = tokenizer(
          remove_html(text),
          return_tensors="pt",
          truncation=True,
          padding="max_length",
          max_length=1024
    ).to(device)

    generated_tokens = model.generate(**inputs, forced_bos_token_id=tokenizer.lang_code_to_id["en_XX"])

    translation = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

    translated_list.append({
        "title": item["title"],
        "translation": translation
    })

print("Feed items imported successfully.")


In [ ]:
print(tokenizer.lang_code_to_id)

In [ ]:
en_id = tokenizer.lang_code_to_id["en_XX"]
print("en_XX ID:", en_id)
print("Model vocab size:", model.config.vocab_size)

In [ ]:
print(len(tokenizer)) 
print(model.get_input_embeddings().weight.shape) 


In [ ]:
print(inputs['input_ids'])
print(inputs['input_ids'].max())
print(model.config.vocab_size)

In [ ]:
item['content']

In [ ]:
with open("translated_output.json", "w", encoding="utf-8") as f:
    json.dump(translated_list, f, ensure_ascii=False, indent=4)